# NutriFile 99 - Smoke Test (no training required)

**Purpose:** verify the full pipeline works end-to-end **without** training any custom models, so you can catch wiring/config issues in seconds instead of hours.

How it works:

1. Loads `yolov8s-seg.pt` pretrained on COCO (auto-downloaded by ultralytics, contains banana/apple/sandwich/orange/broccoli/carrot/hot_dog/pizza/donut/cake).
2. Runs detection on a sample plate photo (downloaded from a public URL OR a path you provide).
3. For each detection, maps the COCO class to our canonical nutrition key and runs the rest of the pipeline (portion -> nutrition -> recommendation -> explanation).
4. Renders an annotated grid.

If this notebook prints kcal numbers and recommendations, your pipeline is correctly wired. Then move to notebooks 01-05 to train the real fine-tuned models.

In [ ]:
# === NutriFile package bootstrap ===
import os, sys, textwrap, pathlib
WORK = pathlib.Path('/kaggle/working')
if not WORK.exists():  # Colab/local fallback
    WORK = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
PKG = WORK / 'nutrifile'
PKG.mkdir(parents=True, exist_ok=True)
FILES = {
    '__init__.py': r'''
"""
NutriFile — end-to-end food computer vision package.

Pipeline:
  image -> produce segmentation (YOLOv8-seg, 1 class)
        -> per-crop ingredient classifier (51 classes)  +  dish classifier (101 classes)
        -> portion estimation (mask area + density)
        -> nutrition aggregation
        -> rule-based recommendation + explanation
"""

from . import config
from . import ontology
from . import nutrition
from . import portion
from . import classifier
from . import pipeline
from . import recommend
from . import explain

__all__ = [
    "config",
    "ontology",
    "nutrition",
    "portion",
    "classifier",
    "pipeline",
    "recommend",
    "explain",
]

'''.lstrip('\n'),
    'config.py': r'''
from __future__ import annotations

import os
from pathlib import Path


# deteksi platform (kaggle, colab, atau lokal)

def detect_platform() -> str:
    """Return 'kaggle', 'colab', atau 'local'."""
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").exists():
        return "kaggle"
    if "COLAB_GPU" in os.environ or Path("/content").exists():
        return "colab"
    return "local"


PLATFORM = detect_platform()


# folder root

if PLATFORM == "kaggle":
    DATA_ROOT = Path("/kaggle/working")
    WORK_ROOT = Path("/kaggle/working")
elif PLATFORM == "colab":
    DATA_ROOT = Path("/content")
    WORK_ROOT = Path("/content")
else:
    # local fallback (e.g. running tests on a dev box)
    DATA_ROOT = Path.home() / "nutrifile_data"
    WORK_ROOT = Path.home() / "nutrifile_work"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)


# lokasi dataset mentah
#
# di Kaggle, dataset yang di-attach lewat "+ Add Data" ada di /kaggle/input/<slug>/
# (read-only). Kalau tidak ada, fallback ke DATA_ROOT.

_KAGGLE_INPUT = Path("/kaggle/input")


def _resolve_dataset(*candidates: Path) -> Path:
    """Return candidate path pertama yang ada; kalau tidak ada, pakai yang terakhir."""
    for c in candidates:
        if c.exists():
            return c
    return candidates[-1]


RAW_FOOD101 = _resolve_dataset(
    _KAGGLE_INPUT / "food41",
    DATA_ROOT / "food101",
)
RAW_PACKEAT = _resolve_dataset(
    _KAGGLE_INPUT / "packed-fruits-and-vegetables-recognition-benchmark",
    DATA_ROOT / "packed-fruits-and-vegetable",
)
RAW_SINGULAR = _resolve_dataset(
    _KAGGLE_INPUT / "singular-food-items",
    DATA_ROOT / "singular-food-items",
)


# dataset yang sudah di-preprocess (siap training)

PRODUCE_SEG_DIR = WORK_ROOT / "produce_dataset"          # YOLOv8-seg, 1 class
INGREDIENT_CLS_DIR = WORK_ROOT / "ingredient_cls_dataset"  # ImageFolder, 51 classes
DISH_CLS_DIR = WORK_ROOT / "dish_cls_dataset"              # ImageFolder, 101 classes


# output dan artefak training

RUNS_DIR = WORK_ROOT / "runs"
WEIGHTS_DIR = WORK_ROOT / "weights"
NUTRITION_DB_PATH = WORK_ROOT / "nutrition_db.json"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)


# nama file bobot model (dipakai di seluruh codebase)

PRODUCE_DETECTOR_WEIGHTS = WEIGHTS_DIR / "produce_yolov8s_seg.pt"
INGREDIENT_CLS_WEIGHTS = WEIGHTS_DIR / "ingredient_effnetb0.pt"
DISH_CLS_WEIGHTS = WEIGHTS_DIR / "dish_effnetb0.pt"


# hyperparameter training (satu sumber kebenaran)

SEED = 42

PRODUCE_TRAIN = dict(
    model_variant="yolov8s-seg.pt",
    epochs=40,
    imgsz=640,
    batch=16,
    patience=8,
    optimizer="auto",
    mosaic=1.0,
    close_mosaic=10,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
)

CLS_TRAIN = dict(
    backbone="efficientnet_b0",
    image_size=224,
    batch_size=64,
    epochs=15,
    lr=3e-4,
    weight_decay=1e-4,
    label_smoothing=0.05,
    mixup=0.1,
    num_workers=4,
)


# threshold inferensi

PRODUCE_CONF_THRESHOLD = 0.15    # was 0.30 -- detector was suppressing real food
PRODUCE_IOU_THRESHOLD = 0.55
INGREDIENT_CONF_FLOOR = 0.55     # was 0.40 -- ingredient head was winning the router too easily
DISH_CONF_FLOOR = 0.45

# When mask covers >= DISH_MASK_AREA_RATIO of the image, prefer dish classifier
# (a single large region is usually a prepared meal, not a single ingredient).
DISH_MASK_AREA_RATIO = 0.10      # was 0.30 -- plated food regions are usually smaller

# deduplikasi deteksi bersarang
#
# NMS standar menekan box yang overlap berdasarkan IoU, tapi tidak
# menangkap kasus di mana satu deteksi SEPENUHNYA MEMUAT deteksi lain
# (misal: pizza utuh DAN sepotong kecil di dalamnya). Tanpa ini,
# total nutrisi terhitung dua kali.
#
# Deteksi kecil D' dibuang kalau salah satu terpenuhi:
#   - containment(D', D) = area(D n D') / area(D')  >=  CONTAINMENT_THRESHOLD
#   - iou(D', D)                                    >=  IOU_THRESHOLD
# D = deteksi besar yang sudah di-keep (opsional same-class only).

DEDUPE_ENABLED = True
DEDUPE_CONTAINMENT_THRESHOLD = 0.70
DEDUPE_IOU_THRESHOLD = 0.60
DEDUPE_SAME_CLASS_ONLY = True


def summary() -> str:
    """Ringkasan konfigurasi yang sedang aktif, buat logging."""
    return (
        f"NutriFile config\n"
        f"  platform           : {PLATFORM}\n"
        f"  DATA_ROOT          : {DATA_ROOT}\n"
        f"  WORK_ROOT          : {WORK_ROOT}\n"
        f"  RAW_FOOD101        : {RAW_FOOD101}\n"
        f"  RAW_PACKEAT        : {RAW_PACKEAT}\n"
        f"  RAW_SINGULAR       : {RAW_SINGULAR}\n"
        f"  PRODUCE_SEG_DIR    : {PRODUCE_SEG_DIR}\n"
        f"  INGREDIENT_CLS_DIR : {INGREDIENT_CLS_DIR}\n"
        f"  DISH_CLS_DIR       : {DISH_CLS_DIR}\n"
        f"  WEIGHTS_DIR        : {WEIGHTS_DIR}\n"
        f"  NUTRITION_DB_PATH  : {NUTRITION_DB_PATH}\n"
    )

'''.lstrip('\n'),
    'ontology.py': r'''
"""
Pemetaan nama kelas untuk NutriFile.

Tiga output classifier yang perlu nama kanonik:
  - PRODUCE         : 1 kelas, segmentasi biner (yolov8-seg).
  - INGREDIENT_*    : 51 kelas bahan mentah dari Singular Food Items.
  - DISH_*          : 101 kelas hidangan matang dari Food101.

Setiap nama kanonik adalah key snake_case yang dipakai database nutrisi.
Ingredient dan dish TIDAK digabung: "rice (cooked)" dan "rice (raw)"
beda secara nutrisi, jadi tetap key terpisah.

Kelas "noise" dari Singular dihapus karena bukan makanan.
"""

from __future__ import annotations

from typing import Dict, List


# ----------------------------------------------------------------------
# Singular Food Items: 51 folders -> 50 ingredient classes (drop "noise")
# Verified from leaf folder listing during dataset audit.
# ----------------------------------------------------------------------

SINGULAR_FOLDERS_RAW: List[str] = [
    "apples", "avocados", "bacon", "bagels", "banana", "beans", "beef",
    "blackberries", "bread", "broccoli", "butter", "cabbage", "carrots",
    "cauliflower", "celery", "cheese", "chicken", "coconut", "corn", "crab",
    "cranberries", "eggs", "fish", "ham", "honey", "lemons", "lettuce",
    "limes", "mangos", "milk", "mushrooms", "noise", "onions", "peppers",
    "potatoes", "raspberries", "rhubarb", "rice", "sausages", "spinach",
    "strawberries", "sweetpotato", "tofu", "tomato", "watermelon", "yogurt",
    # Some Singular dumps include the following additional ingredient folders;
    # we keep them if present. (The audit showed 51 leaf folders.)
    "blueberries", "garlic", "ginger", "olives", "pasta",
]

# Folders that are not food and should be removed before training.
SINGULAR_BLACKLIST = {"noise"}


def build_ingredient_classes(available_folders: List[str]) -> List[str]:
    """
    Given the actual folders present on disk, return a deterministic,
    blacklist-filtered, sorted list of ingredient class names.

    The order returned here IS the class-index order the classifier learns,
    so it must be reproducible (sorted) and stable across runs.
    """
    cleaned = sorted(
        set(available_folders) - SINGULAR_BLACKLIST,
        key=str.lower,
    )
    return cleaned


# Canonical snake_case ingredient keys used by the nutrition DB.
# Mostly identical to the folder name; only a few normalizations.
INGREDIENT_NORMALIZATION: Dict[str, str] = {
    "apples": "apple",
    "avocados": "avocado",
    "carrots": "carrot",
    "eggs": "egg",
    "lemons": "lemon",
    "limes": "lime",
    "mangos": "mango",
    "onions": "onion",
    "peppers": "pepper",
    "potatoes": "potato",
    "raspberries": "raspberry",
    "blackberries": "blackberry",
    "cranberries": "cranberry",
    "strawberries": "strawberry",
    "blueberries": "blueberry",
    "tomato": "tomato",
    "sweetpotato": "sweet_potato",
    "sausages": "sausage",
    "olives": "olive",
}


def ingredient_to_canonical(folder_name: str) -> str:
    """Map a Singular folder name to its canonical snake_case key."""
    name = folder_name.lower().strip()
    return INGREDIENT_NORMALIZATION.get(name, name)


# ----------------------------------------------------------------------
# Food101: 101 fixed dish names. We use the original dataset names as-is.
# These are already snake_case (e.g. apple_pie, beef_carpaccio).
# ----------------------------------------------------------------------

FOOD101_CLASSES: List[str] = [
    "apple_pie", "baby_back_ribs", "baklava", "beef_carpaccio", "beef_tartare",
    "beet_salad", "beignets", "bibimbap", "bread_pudding", "breakfast_burrito",
    "bruschetta", "caesar_salad", "cannoli", "caprese_salad", "carrot_cake",
    "ceviche", "cheesecake", "cheese_plate", "chicken_curry", "chicken_quesadilla",
    "chicken_wings", "chocolate_cake", "chocolate_mousse", "churros", "clam_chowder",
    "club_sandwich", "crab_cakes", "creme_brulee", "croque_madame", "cup_cakes",
    "deviled_eggs", "donuts", "dumplings", "edamame", "eggs_benedict", "escargots",
    "falafel", "filet_mignon", "fish_and_chips", "foie_gras", "french_fries",
    "french_onion_soup", "french_toast", "fried_calamari", "fried_rice",
    "frozen_yogurt", "garlic_bread", "gnocchi", "greek_salad",
    "grilled_cheese_sandwich", "grilled_salmon", "guacamole", "gyoza", "hamburger",
    "hot_and_sour_soup", "hot_dog", "huevos_rancheros", "hummus", "ice_cream",
    "lasagna", "lobster_bisque", "lobster_roll_sandwich", "macaroni_and_cheese",
    "macarons", "miso_soup", "mussels", "nachos", "omelette", "onion_rings",
    "oysters", "pad_thai", "paella", "pancakes", "panna_cotta", "peking_duck",
    "pho", "pizza", "pork_chop", "poutine", "prime_rib", "pulled_pork_sandwich",
    "ramen", "ravioli", "red_velvet_cake", "risotto", "samosa", "sashimi",
    "scallops", "seaweed_salad", "shrimp_and_grits", "spaghetti_bolognese",
    "spaghetti_carbonara", "spring_rolls", "steak", "strawberry_shortcake",
    "sushi", "tacos", "takoyaki", "tiramisu", "tuna_tartare", "waffles",
]
assert len(FOOD101_CLASSES) == 101, f"Expected 101 dishes, got {len(FOOD101_CLASSES)}"


def dish_to_canonical(class_name: str) -> str:
    """Dish names are already canonical; this is a passthrough for symmetry."""
    return class_name


# ----------------------------------------------------------------------
# Routing helpers
# ----------------------------------------------------------------------

def is_dish_key(key: str) -> bool:
    return key in set(FOOD101_CLASSES)


def is_ingredient_key(key: str) -> bool:
    return not is_dish_key(key)

'''.lstrip('\n'),
    'nutrition.py': r'''
"""
Database nutrisi untuk NutriFile.

Semua nilai per 100g porsi yang bisa dimakan. Sumber:
  - USDA FoodData Central (dibulatkan ke integer untuk kkal/natrium,
    1 desimal untuk makro).
  - Untuk hidangan komposit (Food101), dirata-rata dari kartu resep umum
    dan diberi tag kepercayaan.

Skema:
    {
      "<key_kanonik>": {
        "kcal":   <float>,    # per 100g
        "carbs":  <float g>,
        "protein":<float g>,
        "fat":    <float g>,
        "sugar":  <float g>,
        "sodium": <float mg>,
        "fiber":  <float g>,
        "density_g_per_cm2": <float>,   # dipakai oleh estimator porsi
        "confidence": "high" | "med" | "low",
        "category": "ingredient" | "dish",
      }
    }

Key yang tidak ada di database akan fallback ke FALLBACK_ENTRY supaya pipeline tidak crash.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, Optional

from . import ontology


# ----------------------------------------------------------------------
# Reference: typical food density on a plate (grams per cm^2 of visible area).
# Empirical: 1 cm^2 of food viewed from above weighs roughly density g.
# Tuned so a 25 cm plate fully covered in rice weighs ~150–200 g.
# ----------------------------------------------------------------------

_INGREDIENT_DB: Dict[str, Dict[str, float]] = {
    # Fruits
    "apple":       dict(kcal=52,  carbs=14,  protein=0.3, fat=0.2, sugar=10.4, sodium=1,  fiber=2.4, density_g_per_cm2=0.40),
    "avocado":     dict(kcal=160, carbs=8.5, protein=2.0, fat=14.7, sugar=0.7, sodium=7,  fiber=6.7, density_g_per_cm2=0.50),
    "banana":      dict(kcal=89,  carbs=23,  protein=1.1, fat=0.3, sugar=12.2, sodium=1,  fiber=2.6, density_g_per_cm2=0.45),
    "blackberry":  dict(kcal=43,  carbs=10,  protein=1.4, fat=0.5, sugar=4.9,  sodium=1,  fiber=5.3, density_g_per_cm2=0.30),
    "blueberry":   dict(kcal=57,  carbs=14,  protein=0.7, fat=0.3, sugar=10,   sodium=1,  fiber=2.4, density_g_per_cm2=0.30),
    "coconut":     dict(kcal=354, carbs=15,  protein=3.3, fat=33.5, sugar=6.2, sodium=20, fiber=9.0, density_g_per_cm2=0.50),
    "cranberry":   dict(kcal=46,  carbs=12,  protein=0.4, fat=0.1, sugar=4.0,  sodium=2,  fiber=4.6, density_g_per_cm2=0.30),
    "lemon":       dict(kcal=29,  carbs=9,   protein=1.1, fat=0.3, sugar=2.5,  sodium=2,  fiber=2.8, density_g_per_cm2=0.40),
    "lime":        dict(kcal=30,  carbs=11,  protein=0.7, fat=0.2, sugar=1.7,  sodium=2,  fiber=2.8, density_g_per_cm2=0.40),
    "mango":       dict(kcal=60,  carbs=15,  protein=0.8, fat=0.4, sugar=13.7, sodium=1,  fiber=1.6, density_g_per_cm2=0.45),
    "olive":       dict(kcal=115, carbs=6.3, protein=0.8, fat=10.7, sugar=0.5, sodium=735,fiber=3.2, density_g_per_cm2=0.40),
    "raspberry":   dict(kcal=52,  carbs=12,  protein=1.2, fat=0.7, sugar=4.4,  sodium=1,  fiber=6.5, density_g_per_cm2=0.30),
    "rhubarb":     dict(kcal=21,  carbs=4.5, protein=0.9, fat=0.2, sugar=1.1,  sodium=4,  fiber=1.8, density_g_per_cm2=0.30),
    "strawberry":  dict(kcal=32,  carbs=7.7, protein=0.7, fat=0.3, sugar=4.9,  sodium=1,  fiber=2.0, density_g_per_cm2=0.30),
    "watermelon":  dict(kcal=30,  carbs=7.6, protein=0.6, fat=0.2, sugar=6.2,  sodium=1,  fiber=0.4, density_g_per_cm2=0.40),

    # Vegetables
    "broccoli":    dict(kcal=34,  carbs=7.0, protein=2.8, fat=0.4, sugar=1.7,  sodium=33, fiber=2.6, density_g_per_cm2=0.30),
    "cabbage":     dict(kcal=25,  carbs=5.8, protein=1.3, fat=0.1, sugar=3.2,  sodium=18, fiber=2.5, density_g_per_cm2=0.30),
    "carrot":      dict(kcal=41,  carbs=10,  protein=0.9, fat=0.2, sugar=4.7,  sodium=69, fiber=2.8, density_g_per_cm2=0.40),
    "cauliflower": dict(kcal=25,  carbs=5.0, protein=1.9, fat=0.3, sugar=1.9,  sodium=30, fiber=2.0, density_g_per_cm2=0.30),
    "celery":      dict(kcal=16,  carbs=3.0, protein=0.7, fat=0.2, sugar=1.3,  sodium=80, fiber=1.6, density_g_per_cm2=0.25),
    "corn":        dict(kcal=86,  carbs=19,  protein=3.3, fat=1.4, sugar=3.2,  sodium=15, fiber=2.7, density_g_per_cm2=0.45),
    "garlic":      dict(kcal=149, carbs=33,  protein=6.4, fat=0.5, sugar=1.0,  sodium=17, fiber=2.1, density_g_per_cm2=0.40),
    "ginger":      dict(kcal=80,  carbs=18,  protein=1.8, fat=0.8, sugar=1.7,  sodium=13, fiber=2.0, density_g_per_cm2=0.40),
    "lettuce":     dict(kcal=15,  carbs=2.9, protein=1.4, fat=0.2, sugar=0.8,  sodium=28, fiber=1.3, density_g_per_cm2=0.20),
    "mushroom":    dict(kcal=22,  carbs=3.3, protein=3.1, fat=0.3, sugar=2.0,  sodium=5,  fiber=1.0, density_g_per_cm2=0.30),
    "mushrooms":   dict(kcal=22,  carbs=3.3, protein=3.1, fat=0.3, sugar=2.0,  sodium=5,  fiber=1.0, density_g_per_cm2=0.30),
    "onion":       dict(kcal=40,  carbs=9.3, protein=1.1, fat=0.1, sugar=4.2,  sodium=4,  fiber=1.7, density_g_per_cm2=0.40),
    "pepper":      dict(kcal=31,  carbs=6.0, protein=1.0, fat=0.3, sugar=4.2,  sodium=4,  fiber=2.1, density_g_per_cm2=0.30),
    "potato":      dict(kcal=77,  carbs=17,  protein=2.0, fat=0.1, sugar=0.8,  sodium=6,  fiber=2.2, density_g_per_cm2=0.45),
    "spinach":     dict(kcal=23,  carbs=3.6, protein=2.9, fat=0.4, sugar=0.4,  sodium=79, fiber=2.2, density_g_per_cm2=0.20),
    "sweet_potato":dict(kcal=86,  carbs=20,  protein=1.6, fat=0.1, sugar=4.2,  sodium=55, fiber=3.0, density_g_per_cm2=0.45),
    "tomato":      dict(kcal=18,  carbs=3.9, protein=0.9, fat=0.2, sugar=2.6,  sodium=5,  fiber=1.2, density_g_per_cm2=0.35),

    # Protein
    "bacon":       dict(kcal=541, carbs=1.4, protein=37,  fat=42,  sugar=0.0,  sodium=1717,fiber=0.0, density_g_per_cm2=0.45),
    "beans":       dict(kcal=127, carbs=23,  protein=8.7, fat=0.5, sugar=0.3,  sodium=6,  fiber=6.4, density_g_per_cm2=0.40),
    "beef":        dict(kcal=250, carbs=0.0, protein=26,  fat=15,  sugar=0.0,  sodium=72, fiber=0.0, density_g_per_cm2=0.55),
    "chicken":     dict(kcal=165, carbs=0.0, protein=31,  fat=3.6, sugar=0.0,  sodium=74, fiber=0.0, density_g_per_cm2=0.50),
    "crab":        dict(kcal=83,  carbs=0.0, protein=18,  fat=1.0, sugar=0.0,  sodium=395,fiber=0.0, density_g_per_cm2=0.45),
    "egg":         dict(kcal=143, carbs=0.7, protein=12.6,fat=9.5, sugar=0.4,  sodium=142,fiber=0.0, density_g_per_cm2=0.45),
    "fish":        dict(kcal=140, carbs=0.0, protein=20,  fat=6.3, sugar=0.0,  sodium=85, fiber=0.0, density_g_per_cm2=0.50),
    "ham":         dict(kcal=145, carbs=1.5, protein=21,  fat=5.5, sugar=0.8,  sodium=1203,fiber=0.0, density_g_per_cm2=0.50),
    "sausage":     dict(kcal=301, carbs=2.0, protein=12,  fat=27,  sugar=0.6,  sodium=698,fiber=0.0, density_g_per_cm2=0.50),
    "tofu":        dict(kcal=76,  carbs=1.9, protein=8.0, fat=4.8, sugar=0.6,  sodium=7,  fiber=0.3, density_g_per_cm2=0.45),

    # Grains / breads / pasta
    "bagel":       dict(kcal=257, carbs=51,  protein=10,  fat=1.5, sugar=5.0,  sodium=439,fiber=2.1, density_g_per_cm2=0.45),
    "bagels":      dict(kcal=257, carbs=51,  protein=10,  fat=1.5, sugar=5.0,  sodium=439,fiber=2.1, density_g_per_cm2=0.45),
    "bread":       dict(kcal=265, carbs=49,  protein=9.0, fat=3.2, sugar=5.0,  sodium=491,fiber=2.7, density_g_per_cm2=0.30),
    "pasta":       dict(kcal=131, carbs=25,  protein=5.0, fat=1.1, sugar=0.6,  sodium=6,  fiber=1.8, density_g_per_cm2=0.40),
    "rice":        dict(kcal=130, carbs=28,  protein=2.7, fat=0.3, sugar=0.1,  sodium=1,  fiber=0.4, density_g_per_cm2=0.45),

    # Dairy / fats / sweet
    "butter":      dict(kcal=717, carbs=0.1, protein=0.9, fat=81,  sugar=0.1,  sodium=11, fiber=0.0, density_g_per_cm2=0.60),
    "cheese":      dict(kcal=402, carbs=1.3, protein=25,  fat=33,  sugar=0.5,  sodium=621,fiber=0.0, density_g_per_cm2=0.55),
    "honey":       dict(kcal=304, carbs=82,  protein=0.3, fat=0.0, sugar=82,   sodium=4,  fiber=0.2, density_g_per_cm2=0.70),
    "milk":        dict(kcal=42,  carbs=5.0, protein=3.4, fat=1.0, sugar=5.0,  sodium=44, fiber=0.0, density_g_per_cm2=0.50),
    "yogurt":      dict(kcal=59,  carbs=3.6, protein=10,  fat=0.4, sugar=3.2,  sodium=36, fiber=0.0, density_g_per_cm2=0.50),
}


_DISH_DB: Dict[str, Dict[str, float]] = {
    # Cooked dishes — typical per-100g; portion sizes vary so calorie totals
    # rely on visible area + density_g_per_cm2 below.
    "apple_pie":               dict(kcal=237, carbs=34, protein=2.4, fat=11,  sugar=18,  sodium=266, fiber=1.4, density_g_per_cm2=0.40),
    "baby_back_ribs":          dict(kcal=292, carbs=0.0,protein=23,  fat=21,  sugar=0.0, sodium=590, fiber=0.0, density_g_per_cm2=0.60),
    "baklava":                 dict(kcal=403, carbs=46, protein=6.0, fat=22,  sugar=33,  sodium=205, fiber=2.0, density_g_per_cm2=0.45),
    "beef_carpaccio":          dict(kcal=180, carbs=2.0,protein=22,  fat=9.0, sugar=0.5, sodium=420, fiber=0.5, density_g_per_cm2=0.45),
    "beef_tartare":            dict(kcal=190, carbs=2.0,protein=21,  fat=10,  sugar=0.5, sodium=400, fiber=0.5, density_g_per_cm2=0.45),
    "beet_salad":              dict(kcal=110, carbs=14, protein=2.5, fat=5.0, sugar=10,  sodium=350, fiber=3.5, density_g_per_cm2=0.30),
    "beignets":                dict(kcal=400, carbs=45, protein=6.0, fat=22,  sugar=15,  sodium=260, fiber=1.5, density_g_per_cm2=0.40),
    "bibimbap":                dict(kcal=160, carbs=22, protein=7.0, fat=5.0, sugar=3.0, sodium=480, fiber=2.5, density_g_per_cm2=0.50),
    "bread_pudding":           dict(kcal=240, carbs=35, protein=6.0, fat=9.0, sugar=20,  sodium=240, fiber=1.0, density_g_per_cm2=0.45),
    "breakfast_burrito":       dict(kcal=215, carbs=25, protein=10,  fat=9.0, sugar=2.0, sodium=620, fiber=2.0, density_g_per_cm2=0.50),
    "bruschetta":              dict(kcal=200, carbs=28, protein=6.0, fat=7.0, sugar=2.5, sodium=380, fiber=2.0, density_g_per_cm2=0.35),
    "caesar_salad":            dict(kcal=158, carbs=8.0,protein=6.0, fat=12,  sugar=2.0, sodium=470, fiber=2.0, density_g_per_cm2=0.30),
    "cannoli":                 dict(kcal=370, carbs=43, protein=7.0, fat=19,  sugar=28,  sodium=180, fiber=1.0, density_g_per_cm2=0.45),
    "caprese_salad":           dict(kcal=145, carbs=4.5,protein=8.0, fat=11,  sugar=3.0, sodium=380, fiber=1.0, density_g_per_cm2=0.30),
    "carrot_cake":             dict(kcal=380, carbs=46, protein=4.5, fat=20,  sugar=30,  sodium=270, fiber=1.5, density_g_per_cm2=0.45),
    "ceviche":                 dict(kcal=110, carbs=4.0,protein=18,  fat=2.0, sugar=2.0, sodium=420, fiber=1.0, density_g_per_cm2=0.40),
    "cheesecake":              dict(kcal=321, carbs=26, protein=5.5, fat=22,  sugar=22,  sodium=210, fiber=0.5, density_g_per_cm2=0.55),
    "cheese_plate":            dict(kcal=380, carbs=4.0,protein=22,  fat=30,  sugar=2.0, sodium=620, fiber=0.5, density_g_per_cm2=0.50),
    "chicken_curry":           dict(kcal=160, carbs=8.0,protein=14,  fat=8.0, sugar=3.0, sodium=480, fiber=1.5, density_g_per_cm2=0.50),
    "chicken_quesadilla":      dict(kcal=270, carbs=22, protein=14,  fat=14,  sugar=2.0, sodium=560, fiber=1.5, density_g_per_cm2=0.50),
    "chicken_wings":           dict(kcal=290, carbs=1.0,protein=27,  fat=20,  sugar=0.5, sodium=600, fiber=0.0, density_g_per_cm2=0.55),
    "chocolate_cake":          dict(kcal=370, carbs=51, protein=4.5, fat=17,  sugar=37,  sodium=300, fiber=2.5, density_g_per_cm2=0.50),
    "chocolate_mousse":        dict(kcal=270, carbs=29, protein=4.5, fat=15,  sugar=24,  sodium=60,  fiber=1.5, density_g_per_cm2=0.50),
    "churros":                 dict(kcal=370, carbs=44, protein=5.0, fat=20,  sugar=15,  sodium=250, fiber=1.5, density_g_per_cm2=0.45),
    "clam_chowder":            dict(kcal=95,  carbs=10, protein=5.0, fat=4.0, sugar=2.0, sodium=480, fiber=1.0, density_g_per_cm2=0.50),
    "club_sandwich":           dict(kcal=290, carbs=22, protein=15,  fat=15,  sugar=3.0, sodium=620, fiber=2.0, density_g_per_cm2=0.40),
    "crab_cakes":              dict(kcal=215, carbs=12, protein=14,  fat=12,  sugar=1.0, sodium=560, fiber=1.0, density_g_per_cm2=0.50),
    "creme_brulee":            dict(kcal=290, carbs=20, protein=4.0, fat=21,  sugar=18,  sodium=70,  fiber=0.0, density_g_per_cm2=0.55),
    "croque_madame":           dict(kcal=300, carbs=20, protein=16,  fat=17,  sugar=2.0, sodium=580, fiber=1.5, density_g_per_cm2=0.50),
    "cup_cakes":               dict(kcal=370, carbs=48, protein=4.0, fat=18,  sugar=35,  sodium=290, fiber=1.0, density_g_per_cm2=0.45),
    "deviled_eggs":            dict(kcal=190, carbs=1.5,protein=11,  fat=15,  sugar=1.0, sodium=300, fiber=0.0, density_g_per_cm2=0.45),
    "donuts":                  dict(kcal=420, carbs=51, protein=5.5, fat=22,  sugar=25,  sodium=320, fiber=1.5, density_g_per_cm2=0.40),
    "dumplings":               dict(kcal=240, carbs=33, protein=8.0, fat=8.0, sugar=2.0, sodium=420, fiber=1.5, density_g_per_cm2=0.50),
    "edamame":                 dict(kcal=121, carbs=9.0,protein=12,  fat=5.0, sugar=2.0, sodium=6,   fiber=5.0, density_g_per_cm2=0.40),
    "eggs_benedict":           dict(kcal=230, carbs=14, protein=11,  fat=15,  sugar=2.0, sodium=440, fiber=1.0, density_g_per_cm2=0.50),
    "escargots":               dict(kcal=200, carbs=2.0,protein=16,  fat=14,  sugar=0.0, sodium=420, fiber=0.5, density_g_per_cm2=0.55),
    "falafel":                 dict(kcal=333, carbs=32, protein=14,  fat=18,  sugar=1.0, sodium=294, fiber=4.9, density_g_per_cm2=0.45),
    "filet_mignon":             dict(kcal=270, carbs=0.0,protein=27,  fat=18,  sugar=0.0, sodium=70,  fiber=0.0, density_g_per_cm2=0.60),
    "fish_and_chips":          dict(kcal=290, carbs=27, protein=11,  fat=15,  sugar=1.0, sodium=520, fiber=2.5, density_g_per_cm2=0.50),
    "foie_gras":               dict(kcal=460, carbs=4.0,protein=12,  fat=44,  sugar=2.0, sodium=700, fiber=0.0, density_g_per_cm2=0.60),
    "french_fries":            dict(kcal=312, carbs=41, protein=3.4, fat=15,  sugar=0.3, sodium=210, fiber=3.8, density_g_per_cm2=0.40),
    "french_onion_soup":       dict(kcal=95,  carbs=9.0,protein=4.0, fat=5.0, sugar=4.0, sodium=560, fiber=1.5, density_g_per_cm2=0.50),
    "french_toast":            dict(kcal=240, carbs=27, protein=8.0, fat=11,  sugar=8.0, sodium=320, fiber=1.0, density_g_per_cm2=0.45),
    "fried_calamari":          dict(kcal=250, carbs=15, protein=14,  fat=14,  sugar=0.5, sodium=480, fiber=0.5, density_g_per_cm2=0.50),
    "fried_rice":              dict(kcal=180, carbs=24, protein=5.0, fat=7.0, sugar=1.0, sodium=480, fiber=1.0, density_g_per_cm2=0.50),
    "frozen_yogurt":           dict(kcal=159, carbs=24, protein=4.0, fat=4.0, sugar=23,  sodium=70,  fiber=0.0, density_g_per_cm2=0.55),
    "garlic_bread":            dict(kcal=350, carbs=43, protein=8.0, fat=15,  sugar=2.0, sodium=540, fiber=2.0, density_g_per_cm2=0.40),
    "gnocchi":                 dict(kcal=170, carbs=32, protein=4.0, fat=2.0, sugar=1.0, sodium=240, fiber=1.5, density_g_per_cm2=0.50),
    "greek_salad":             dict(kcal=120, carbs=6.0,protein=4.0, fat=9.0, sugar=3.0, sodium=480, fiber=2.0, density_g_per_cm2=0.30),
    "grilled_cheese_sandwich": dict(kcal=320, carbs=28, protein=13,  fat=17,  sugar=4.0, sodium=620, fiber=2.0, density_g_per_cm2=0.45),
    "grilled_salmon":          dict(kcal=208, carbs=0.0,protein=22,  fat=13,  sugar=0.0, sodium=80,  fiber=0.0, density_g_per_cm2=0.55),
    "guacamole":               dict(kcal=150, carbs=8.0,protein=2.0, fat=14,  sugar=1.0, sodium=300, fiber=6.0, density_g_per_cm2=0.55),
    "gyoza":                   dict(kcal=220, carbs=27, protein=7.0, fat=8.0, sugar=2.0, sodium=460, fiber=1.5, density_g_per_cm2=0.50),
    "hamburger":               dict(kcal=295, carbs=22, protein=17,  fat=15,  sugar=4.0, sodium=480, fiber=1.5, density_g_per_cm2=0.50),
    "hot_and_sour_soup":       dict(kcal=70,  carbs=8.0,protein=4.0, fat=2.5, sugar=2.0, sodium=620, fiber=1.0, density_g_per_cm2=0.50),
    "hot_dog":                 dict(kcal=290, carbs=18, protein=10,  fat=20,  sugar=4.0, sodium=680, fiber=1.0, density_g_per_cm2=0.50),
    "huevos_rancheros":        dict(kcal=180, carbs=18, protein=9.0, fat=8.0, sugar=2.0, sodium=420, fiber=2.5, density_g_per_cm2=0.50),
    "hummus":                  dict(kcal=166, carbs=14, protein=8.0, fat=10,  sugar=0.5, sodium=380, fiber=6.0, density_g_per_cm2=0.50),
    "ice_cream":               dict(kcal=207, carbs=24, protein=3.5, fat=11,  sugar=21,  sodium=80,  fiber=0.5, density_g_per_cm2=0.55),
    "lasagna":                 dict(kcal=180, carbs=18, protein=10,  fat=8.0, sugar=3.0, sodium=480, fiber=2.0, density_g_per_cm2=0.55),
    "lobster_bisque":          dict(kcal=110, carbs=6.0,protein=6.0, fat=7.0, sugar=2.0, sodium=540, fiber=0.5, density_g_per_cm2=0.50),
    "lobster_roll_sandwich":   dict(kcal=240, carbs=18, protein=14,  fat=12,  sugar=3.0, sodium=520, fiber=1.0, density_g_per_cm2=0.45),
    "macaroni_and_cheese":     dict(kcal=210, carbs=22, protein=9.0, fat=10,  sugar=2.0, sodium=480, fiber=1.0, density_g_per_cm2=0.50),
    "macarons":                dict(kcal=420, carbs=60, protein=6.0, fat=18,  sugar=45,  sodium=80,  fiber=2.0, density_g_per_cm2=0.50),
    "miso_soup":               dict(kcal=40,  carbs=5.0,protein=2.5, fat=1.0, sugar=1.0, sodium=620, fiber=0.5, density_g_per_cm2=0.50),
    "mussels":                 dict(kcal=172, carbs=7.0,protein=24,  fat=4.5, sugar=0.5, sodium=369, fiber=0.0, density_g_per_cm2=0.55),
    "nachos":                  dict(kcal=343, carbs=36, protein=8.0, fat=18,  sugar=2.0, sodium=560, fiber=3.0, density_g_per_cm2=0.40),
    "omelette":                dict(kcal=154, carbs=1.0,protein=11,  fat=12,  sugar=0.5, sodium=300, fiber=0.0, density_g_per_cm2=0.50),
    "onion_rings":             dict(kcal=330, carbs=39, protein=5.0, fat=17,  sugar=4.0, sodium=440, fiber=2.0, density_g_per_cm2=0.40),
    "oysters":                 dict(kcal=68,  carbs=4.0,protein=7.0, fat=2.5, sugar=0.0, sodium=240, fiber=0.0, density_g_per_cm2=0.55),
    "pad_thai":                dict(kcal=180, carbs=24, protein=7.0, fat=6.0, sugar=4.0, sodium=480, fiber=1.5, density_g_per_cm2=0.50),
    "paella":                  dict(kcal=170, carbs=20, protein=8.0, fat=6.0, sugar=1.5, sodium=460, fiber=1.5, density_g_per_cm2=0.50),
    "pancakes":                dict(kcal=227, carbs=28, protein=6.4, fat=10,  sugar=6.0, sodium=439, fiber=1.0, density_g_per_cm2=0.45),
    "panna_cotta":             dict(kcal=240, carbs=22, protein=4.0, fat=15,  sugar=20,  sodium=60,  fiber=0.0, density_g_per_cm2=0.55),
    "peking_duck":             dict(kcal=337, carbs=2.0,protein=19,  fat=28,  sugar=1.0, sodium=84,  fiber=0.0, density_g_per_cm2=0.55),
    "pho":                     dict(kcal=85,  carbs=12, protein=5.5, fat=1.5, sugar=2.0, sodium=620, fiber=1.0, density_g_per_cm2=0.50),
    "pizza":                   dict(kcal=266, carbs=33, protein=11,  fat=10,  sugar=3.6, sodium=598, fiber=2.3, density_g_per_cm2=0.45),
    "pork_chop":               dict(kcal=231, carbs=0.0,protein=26,  fat=14,  sugar=0.0, sodium=60,  fiber=0.0, density_g_per_cm2=0.55),
    "poutine":                 dict(kcal=233, carbs=25, protein=6.0, fat=12,  sugar=1.0, sodium=480, fiber=2.0, density_g_per_cm2=0.50),
    "prime_rib":               dict(kcal=320, carbs=0.0,protein=25,  fat=24,  sugar=0.0, sodium=80,  fiber=0.0, density_g_per_cm2=0.60),
    "pulled_pork_sandwich":    dict(kcal=265, carbs=22, protein=15,  fat=12,  sugar=8.0, sodium=580, fiber=1.5, density_g_per_cm2=0.50),
    "ramen":                   dict(kcal=140, carbs=18, protein=5.0, fat=5.0, sugar=2.0, sodium=860, fiber=1.0, density_g_per_cm2=0.50),
    "ravioli":                 dict(kcal=180, carbs=24, protein=8.0, fat=6.0, sugar=2.0, sodium=420, fiber=1.5, density_g_per_cm2=0.50),
    "red_velvet_cake":         dict(kcal=370, carbs=49, protein=4.0, fat=18,  sugar=36,  sodium=290, fiber=1.0, density_g_per_cm2=0.50),
    "risotto":                 dict(kcal=175, carbs=25, protein=4.0, fat=6.0, sugar=1.0, sodium=440, fiber=1.0, density_g_per_cm2=0.50),
    "samosa":                  dict(kcal=308, carbs=32, protein=5.0, fat=18,  sugar=2.0, sodium=420, fiber=2.5, density_g_per_cm2=0.45),
    "sashimi":                 dict(kcal=130, carbs=0.0,protein=22,  fat=5.0, sugar=0.0, sodium=80,  fiber=0.0, density_g_per_cm2=0.55),
    "scallops":                dict(kcal=88,  carbs=2.0,protein=17,  fat=0.8, sugar=0.0, sodium=160, fiber=0.0, density_g_per_cm2=0.55),
    "seaweed_salad":           dict(kcal=70,  carbs=8.0,protein=2.0, fat=4.0, sugar=2.0, sodium=620, fiber=2.5, density_g_per_cm2=0.30),
    "shrimp_and_grits":        dict(kcal=200, carbs=20, protein=11,  fat=9.0, sugar=1.5, sodium=520, fiber=1.0, density_g_per_cm2=0.50),
    "spaghetti_bolognese":     dict(kcal=160, carbs=20, protein=8.0, fat=5.0, sugar=3.0, sodium=420, fiber=2.0, density_g_per_cm2=0.50),
    "spaghetti_carbonara":     dict(kcal=215, carbs=22, protein=9.0, fat=10,  sugar=1.5, sodium=520, fiber=1.5, density_g_per_cm2=0.50),
    "spring_rolls":            dict(kcal=190, carbs=22, protein=4.0, fat=9.0, sugar=2.0, sodium=380, fiber=1.5, density_g_per_cm2=0.45),
    "steak":                   dict(kcal=271, carbs=0.0,protein=26,  fat=18,  sugar=0.0, sodium=60,  fiber=0.0, density_g_per_cm2=0.60),
    "strawberry_shortcake":    dict(kcal=290, carbs=37, protein=4.0, fat=14,  sugar=24,  sodium=240, fiber=1.5, density_g_per_cm2=0.45),
    "sushi":                   dict(kcal=150, carbs=22, protein=6.0, fat=4.0, sugar=2.5, sodium=420, fiber=1.0, density_g_per_cm2=0.55),
    "tacos":                   dict(kcal=226, carbs=21, protein=9.0, fat=12,  sugar=2.0, sodium=410, fiber=2.5, density_g_per_cm2=0.45),
    "takoyaki":                dict(kcal=185, carbs=18, protein=8.0, fat=8.0, sugar=2.0, sodium=460, fiber=1.0, density_g_per_cm2=0.50),
    "tiramisu":                dict(kcal=340, carbs=30, protein=6.0, fat=22,  sugar=22,  sodium=170, fiber=0.5, density_g_per_cm2=0.55),
    "tuna_tartare":            dict(kcal=170, carbs=2.0,protein=24,  fat=7.0, sugar=0.5, sodium=320, fiber=0.5, density_g_per_cm2=0.50),
    "waffles":                 dict(kcal=291, carbs=33, protein=8.0, fat=15,  sugar=8.0, sodium=511, fiber=1.5, density_g_per_cm2=0.45),
}


# ----------------------------------------------------------------------
# Fallback for any missing canonical key.
# Values picked so that "unknown" food contributes a moderate generic load.
# ----------------------------------------------------------------------

FALLBACK_ENTRY: Dict[str, float] = dict(
    kcal=200, carbs=20, protein=8.0, fat=8.0,
    sugar=3.0, sodium=300, fiber=2.0, density_g_per_cm2=0.45,
)


# ----------------------------------------------------------------------
# Tag categories + confidence
# ----------------------------------------------------------------------

def _tag(entry: Dict[str, float], category: str, confidence: str) -> Dict[str, float]:
    out = dict(entry)
    out["category"] = category
    out["confidence"] = confidence
    return out


_DB_FINAL: Dict[str, Dict[str, float]] = {}

for key, entry in _INGREDIENT_DB.items():
    _DB_FINAL[key] = _tag(entry, "ingredient", "high")

for key, entry in _DISH_DB.items():
    _DB_FINAL[key] = _tag(entry, "dish", "med")


def lookup(canonical_key: str) -> Dict[str, float]:
    """Return nutrition entry for a key; fallback if missing."""
    if canonical_key in _DB_FINAL:
        return _DB_FINAL[canonical_key]
    return _tag(FALLBACK_ENTRY, "unknown", "low")


def has_entry(canonical_key: str) -> bool:
    return canonical_key in _DB_FINAL


def all_keys() -> list[str]:
    return sorted(_DB_FINAL.keys())


def coverage_report() -> Dict[str, list[str]]:
    """Report which ontology keys do/do not have nutrition entries."""
    expected = set(ontology.FOOD101_CLASSES)
    expected |= {
        ontology.ingredient_to_canonical(name)
        for name in ontology.SINGULAR_FOLDERS_RAW
        if name not in ontology.SINGULAR_BLACKLIST
    }
    have = set(_DB_FINAL.keys())
    return {
        "covered": sorted(expected & have),
        "missing": sorted(expected - have),
        "extra":   sorted(have - expected),
    }


def export_json(path: Path | str) -> Path:
    """Persist the database to disk for reuse outside the package."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(_DB_FINAL, f, indent=2, sort_keys=True)
    return path


def load_json(path: Path | str) -> Dict[str, Dict[str, float]]:
    """Override the in-memory DB from a JSON file (e.g. an updated copy)."""
    global _DB_FINAL
    with Path(path).open("r", encoding="utf-8") as f:
        _DB_FINAL = json.load(f)
    return _DB_FINAL

'''.lstrip('\n'),
    'portion.py': r'''
"""
Estimasi porsi makanan dari mask segmentasi.

Caranya:
  1. Asumsikan foto dari atas, di atas permukaan makan biasa.
  2. Ambil area polygon mask (koordinat ternormalisasi).
  3. Kalikan dengan area piring referensi (cm2). Default: piring 25 cm
     -> pi * 12.5^2 = 490 cm2, sekitar 70% dari area frame tipikal.
  4. Kalikan dengan density_g_per_cm2 spesifik makanan dari DB nutrisi.

Kalau ada objek referensi (koin, tangan, dll), langkah 3 di-override.

Sengaja dibuat sederhana -- tanpa depth estimation atau rekonstruksi 3D.
Cukup akurat untuk memberikan angka nutrisi yang masuk akal.
"""

from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Optional, Sequence

import numpy as np


# nilai default

# 25 cm diameter standard dinner plate, ~70% frame coverage.
_REF_PLATE_AREA_CM2 = math.pi * (25 / 2) ** 2  # ≈ 490.87 cm^2
_REF_FRAME_FRACTION = 0.70                      # plate ≈ 70% of image area

DEFAULT_FRAME_AREA_CM2 = _REF_PLATE_AREA_CM2 / _REF_FRAME_FRACTION  # ≈ 701 cm^2


@dataclass
class PortionEstimate:
    grams: float
    area_cm2: float
    mask_area_ratio: float            # 0..1 mask coverage of frame
    density_used: float               # g per cm^2
    method: str                       # "default" | "reference_object"

    def to_dict(self) -> dict:
        return {
            "grams": round(self.grams, 1),
            "area_cm2": round(self.area_cm2, 1),
            "mask_area_ratio": round(self.mask_area_ratio, 4),
            "density_g_per_cm2": round(self.density_used, 3),
            "method": self.method,
        }


# ----------------------------------------------------------------------
# Geometry helpers
# ----------------------------------------------------------------------

def polygon_area_normalized(points_xy: Sequence[Sequence[float]]) -> float:
    """
    Shoelace area for a polygon with normalized (0..1) coordinates.
    Returns area in normalized units (0..1).
    """
    pts = np.asarray(points_xy, dtype=np.float64)
    if pts.ndim != 2 or pts.shape[0] < 3:
        return 0.0
    x = pts[:, 0]
    y = pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def mask_area_ratio_from_binary(mask: np.ndarray) -> float:
    """Fraction of pixels = 1 in a 2-D binary/uint8 mask."""
    if mask.size == 0:
        return 0.0
    return float((mask > 0).sum()) / float(mask.size)


# ----------------------------------------------------------------------
# Estimator
# ----------------------------------------------------------------------

def estimate_grams(
    mask_area_ratio: float,
    density_g_per_cm2: float,
    frame_area_cm2: Optional[float] = None,
    reference_object_cm2: Optional[float] = None,
    reference_object_mask_ratio: Optional[float] = None,
) -> PortionEstimate:
    """
    Convert a mask area ratio + density into grams.

    Parameters
    ----------
    mask_area_ratio
        Object mask fraction (0..1) of the full image area.
    density_g_per_cm2
        From nutrition DB.
    frame_area_cm2
        Optional override of the total physical area visible in the frame.
        Default = ~701 cm^2 (25 cm plate filling 70% of frame).
    reference_object_cm2, reference_object_mask_ratio
        If both supplied, the physical frame area is back-computed
        (reference_object_cm2 / reference_object_mask_ratio).
    """
    if reference_object_cm2 is not None and reference_object_mask_ratio:
        frame_area_cm2 = reference_object_cm2 / reference_object_mask_ratio
        method = "reference_object"
    else:
        if frame_area_cm2 is None:
            frame_area_cm2 = DEFAULT_FRAME_AREA_CM2
        method = "default"

    area_cm2 = mask_area_ratio * frame_area_cm2
    grams = area_cm2 * density_g_per_cm2

    return PortionEstimate(
        grams=grams,
        area_cm2=area_cm2,
        mask_area_ratio=mask_area_ratio,
        density_used=density_g_per_cm2,
        method=method,
    )

'''.lstrip('\n'),
    'classifier.py': r'''
"""
Wrapper untuk klasifier EfficientNet-B0 (bahan maupun hidangan).

Kedua head pakai arsitektur yang sama, cuma beda jumlah kelas di layer FC terakhir.
Bobot disimpan sebagai state_dict biasa + file JSON kecil berisi daftar nama kelas
sesuai urutan index waktu training.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Sequence

import numpy as np


# torch dan torchvision berat, jadi di-import lazy supaya startup cepat

def _torch():
    import torch  # noqa: WPS433
    return torch


def _torchvision():
    import torchvision  # noqa: WPS433
    return torchvision


def _transforms_for_inference(image_size: int):
    tv = _torchvision()
    return tv.transforms.Compose([
        tv.transforms.Resize(int(image_size * 1.15)),
        tv.transforms.CenterCrop(image_size),
        tv.transforms.ToTensor(),
        tv.transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]),
    ])


def build_efficientnet_b0(num_classes: int, pretrained: bool = True):
    """Buat EfficientNet-B0 dengan layer classifier terakhir di-resize."""
    torch = _torch()
    tv = _torchvision()
    weights = tv.models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = tv.models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(in_features, num_classes)
    return model


# simpan / muat model

@dataclass
class ClassifierBundle:
    """Model + nama kelas + transform, siap pakai."""
    model: "torch.nn.Module"  # noqa: F821
    class_names: List[str]
    image_size: int
    device: str

    def save(self, weights_path: Path, meta_path: Optional[Path] = None) -> None:
        torch = _torch()
        weights_path = Path(weights_path)
        weights_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.model.state_dict(), weights_path)

        if meta_path is None:
            meta_path = weights_path.with_suffix(".classes.json")
        meta = {"class_names": self.class_names, "image_size": self.image_size}
        Path(meta_path).write_text(json.dumps(meta, indent=2))


def load_classifier(
    weights_path: Path,
    image_size: int = 224,
    device: Optional[str] = None,
) -> ClassifierBundle:
    """Muat klasifier yang sudah di-train dari file .pt + .classes.json."""
    torch = _torch()
    weights_path = Path(weights_path)
    meta_path = weights_path.with_suffix(".classes.json")
    meta = json.loads(meta_path.read_text())
    class_names: List[str] = list(meta["class_names"])
    img_size = int(meta.get("image_size", image_size))

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = build_efficientnet_b0(num_classes=len(class_names), pretrained=False)
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state)
    model.to(device).eval()

    return ClassifierBundle(model=model, class_names=class_names,
                            image_size=img_size, device=device)


# fungsi inferensi

def _to_pil(image_bgr_or_rgb_or_pil):
    """Terima np.ndarray (BGR/RGB) atau PIL.Image, kembalikan PIL RGB."""
    from PIL import Image as PILImage
    if isinstance(image_bgr_or_rgb_or_pil, PILImage.Image):
        return image_bgr_or_rgb_or_pil.convert("RGB")
    arr = np.asarray(image_bgr_or_rgb_or_pil)
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    if arr.shape[2] == 4:
        arr = arr[:, :, :3]
    # If looks BGR (cv2), swap. We can't be 100% sure; heuristic: assume RGB
    # only when channel 0 mean < channel 2 mean. Not perfect, but the caller
    # is expected to hand RGB anyway.
    return PILImage.fromarray(arr.astype("uint8")).convert("RGB")


def predict_topk(
    bundle: ClassifierBundle,
    image,
    topk: int = 3,
):
    """
    Jalankan inferensi pada satu gambar.
    Return list of (nama_kelas, probabilitas) diurutkan dari yang tertinggi.
    """
    torch = _torch()
    pil = _to_pil(image)
    tfm = _transforms_for_inference(bundle.image_size)
    x = tfm(pil).unsqueeze(0).to(bundle.device)

    with torch.no_grad():
        logits = bundle.model(x)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    top_idx = np.argsort(-probs)[:topk]
    return [(bundle.class_names[i], float(probs[i])) for i in top_idx]


def predict_batch_topk(
    bundle: ClassifierBundle,
    images: Sequence,
    topk: int = 3,
    batch_size: int = 32,
):
    """Versi batch dari predict_topk, lebih cepat untuk banyak gambar sekaligus."""
    torch = _torch()
    tfm = _transforms_for_inference(bundle.image_size)
    results = []

    for start in range(0, len(images), batch_size):
        chunk = images[start:start + batch_size]
        tensors = [tfm(_to_pil(img)) for img in chunk]
        batch = torch.stack(tensors).to(bundle.device)

        with torch.no_grad():
            logits = bundle.model(batch)
            probs = torch.softmax(logits, dim=1).cpu().numpy()

        for row in probs:
            top_idx = np.argsort(-row)[:topk]
            results.append([(bundle.class_names[i], float(row[i])) for i in top_idx])

    return results

'''.lstrip('\n'),
    'pipeline.py': r'''
"""
Pipeline inferensi NutriFile dari ujung ke ujung.

Alur:
    gambar
      -> detektor YOLOv8-seg "produce" (biner, polygon mask per-instance)
      -> untuk setiap area yang terdeteksi:
            crop ROI -> klasifier bahan -> top-3 (kelas, prob)
                     -> klasifier hidangan -> top-3 (kelas, prob)
                     -> router pilih label kanonik
      -> estimasi porsi per deteksi (area mask * densitas)
      -> agregasi nutrisi
      -> rekomendasi + penjelasan
"""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Sequence

import numpy as np

from . import config
from . import nutrition
from . import ontology
from . import portion as portion_mod
from . import classifier as cls_mod
from . import recommend as rec_mod
from . import explain as exp_mod


# ----------------------------------------------------------------------
# Lazy imports
# ----------------------------------------------------------------------

def _cv2():
    import cv2  # noqa: WPS433
    return cv2


def _yolo():
    from ultralytics import YOLO  # noqa: WPS433
    return YOLO


# ----------------------------------------------------------------------
# Complex-dish -> simple-ingredient fallback table.
#
# Used by _route_label to detect cases where the dish classifier picks a
# "complex dish that has a plain-ingredient alternative" (e.g. fried_rice
# while looking at plain steamed rice). When the ingredient classifier also
# sees the base ingredient with MODEST confidence (0.15-0.50), we override
# to the simpler interpretation to avoid overestimating calories.
# ----------------------------------------------------------------------
DISH_TO_INGREDIENT_FALLBACK = {
    # Rice family
    "fried_rice": "rice", "risotto": "rice",
    "paella": "rice", "bibimbap": "rice",
    # Bread family
    "garlic_bread": "bread", "french_toast": "bread",
    "bread_pudding": "bread",
    # Egg family
    "omelette": "egg", "eggs_benedict": "egg",
    "deviled_eggs": "egg", "huevos_rancheros": "egg",
    # Chicken family
    "chicken_curry": "chicken", "chicken_wings": "chicken",
    "chicken_quesadilla": "chicken",
    # Potato family
    "french_fries": "potato", "poutine": "potato",
    # Beef family
    "beef_carpaccio": "beef", "beef_tartare": "beef",
    "filet_mignon": "beef", "prime_rib": "beef",
    # Fish family
    "grilled_salmon": "fish", "sashimi": "fish",
    # Salad-style with one dominant veg
    "caprese_salad": "tomato",
}

# Decision thresholds for the family guard
SIMPLE_INGREDIENT_MIN = 0.15
SIMPLE_INGREDIENT_MAX = 0.50

# Reject regions where both heads are unconfident (probably not food at all)
UNKNOWN_REJECT_FLOOR = 0.30


# ----------------------------------------------------------------------
# COCO ensemble: stock yolov8s-seg.pt already knows several food classes.
# Running it alongside the produce detector recovers prepared dishes
# (pizza, donut, cake, hot_dog, sandwich) that the PackEat-trained produce
# detector completely misses.
# ----------------------------------------------------------------------
COCO_FOOD_CLASSES = {
    46: "banana", 47: "apple", 48: "sandwich", 49: "orange",
    50: "broccoli", 51: "carrot", 52: "hot_dog", 53: "pizza",
    54: "donut", 55: "cake",
}

# Map COCO label -> key in nutrition DB
COCO_TO_NUTRITION_KEY = {
    "banana": "banana", "apple": "apple",
    "sandwich": "club_sandwich", "orange": "lemon",
    "broccoli": "broccoli", "carrot": "carrot",
    "hot_dog": "hot_dog", "pizza": "pizza",
    "donut": "donuts", "cake": "chocolate_cake",
}

COCO_CONF_THRESHOLD = 0.25
COCO_IOU_THRESHOLD = 0.55
COCO_VS_PRODUCE_CONTAINMENT = 0.50
COCO_VS_PRODUCE_IOU = 0.40


# ----------------------------------------------------------------------
# Detection result dataclasses
# ----------------------------------------------------------------------

@dataclass
class Detection:
    bbox_xyxy: List[float]             # absolute pixel coordinates
    score: float
    mask_area_ratio: float             # 0..1, mask coverage of full image
    polygon_xy_norm: List[List[float]] # polygon coords in 0..1

    # filled in later
    label: Optional[str] = None
    label_source: Optional[str] = None  # "ingredient" | "dish"
    ingredient_top: List[tuple] = field(default_factory=list)  # [(name, prob), ...]
    dish_top:       List[tuple] = field(default_factory=list)
    portion: Optional[dict] = None
    nutrition: Optional[dict] = None

    def to_dict(self) -> dict:
        return {
            "bbox_xyxy": [round(v, 2) for v in self.bbox_xyxy],
            "score": round(self.score, 4),
            "mask_area_ratio": round(self.mask_area_ratio, 4),
            "label": self.label,
            "label_source": self.label_source,
            "ingredient_top": [(n, round(p, 4)) for n, p in self.ingredient_top],
            "dish_top": [(n, round(p, 4)) for n, p in self.dish_top],
            "portion": self.portion,
            "nutrition": self.nutrition,
        }


@dataclass
class PipelineResult:
    image_shape: tuple
    detections: List[Detection]
    aggregated_nutrition: dict
    recommendations: List[dict]
    explanation: str

    def to_dict(self) -> dict:
        return {
            "image_shape": list(self.image_shape),
            "detections": [d.to_dict() for d in self.detections],
            "aggregated_nutrition": self.aggregated_nutrition,
            "recommendations": self.recommendations,
            "explanation": self.explanation,
        }


# ----------------------------------------------------------------------
# Routing logic between ingredient vs dish classifier
# ----------------------------------------------------------------------

def _route_label(
    detection: Detection,
    ingredient_top: list,
    dish_top: list,
) -> tuple:
    """
    Decide whether the label for this region should be an ingredient or a dish.

    Order of decisions:
      1. If both heads' top-1 prob is very low -> reject as 'unknown'
         (probably not food at all - hand, plate, utensil, etc.)
      2. If dish predicts a "complex dish" that has a plain-ingredient
         alternative AND the ingredient classifier sees the base ingredient
         with MODEST confidence (0.15-0.50) -> override to the simple
         ingredient (prevents 'plain rice -> fried_rice' overestimation).
      3. Big-region + confident dish -> dish.
      4. Confident-enough ingredient that beats dish -> ingredient.
      5. Confident dish (small region) -> dish.
      6. Fallback to whichever head has higher top-1.
    """
    ing_name, ing_prob = ingredient_top[0] if ingredient_top else (None, 0.0)
    dish_name, dish_prob = dish_top[0] if dish_top else (None, 0.0)

    # (1) Non-food reject
    if ing_prob < UNKNOWN_REJECT_FLOOR and dish_prob < UNKNOWN_REJECT_FLOOR:
        return "unknown", "unknown"

    # (2) Family guard: complex-dish -> simple-ingredient override
    if dish_name in DISH_TO_INGREDIENT_FALLBACK:
        expected_ingredient = DISH_TO_INGREDIENT_FALLBACK[dish_name]
        for ing_candidate, ing_candidate_prob in ingredient_top:
            if ing_candidate == expected_ingredient:
                if SIMPLE_INGREDIENT_MIN <= ing_candidate_prob < SIMPLE_INGREDIENT_MAX:
                    return (
                        ontology.ingredient_to_canonical(expected_ingredient),
                        "ingredient_override",
                    )
                break

    big_region = detection.mask_area_ratio >= config.DISH_MASK_AREA_RATIO

    if big_region and dish_prob >= config.DISH_CONF_FLOOR:
        return ontology.dish_to_canonical(dish_name), "dish"

    if ing_prob >= config.INGREDIENT_CONF_FLOOR and ing_prob >= dish_prob:
        return ontology.ingredient_to_canonical(ing_name), "ingredient"

    if dish_prob >= config.DISH_CONF_FLOOR:
        return ontology.dish_to_canonical(dish_name), "dish"

    if ing_prob >= dish_prob and ing_name is not None:
        return ontology.ingredient_to_canonical(ing_name), "ingredient"
    if dish_name is not None:
        return ontology.dish_to_canonical(dish_name), "dish"

    return "unknown", "unknown"


# ----------------------------------------------------------------------
# Detector helpers
# ----------------------------------------------------------------------

def _run_yolo_seg(detector_model, image_rgb: np.ndarray):
    """Run YOLOv8-seg and return a list of Detection objects (no labels yet)."""
    cv2 = _cv2()
    H, W = image_rgb.shape[:2]
    img_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    pred = detector_model.predict(
        source=img_bgr,
        conf=config.PRODUCE_CONF_THRESHOLD,
        iou=config.PRODUCE_IOU_THRESHOLD,
        verbose=False,
    )[0]

    detections: List[Detection] = []
    if pred.masks is None or pred.boxes is None:
        return detections

    boxes_xyxy = pred.boxes.xyxy.cpu().numpy()
    scores = pred.boxes.conf.cpu().numpy()
    polys_xy = pred.masks.xy  # list of arrays (Nx2) in absolute pixel coords

    for box, score, poly_px in zip(boxes_xyxy, scores, polys_xy):
        if poly_px is None or len(poly_px) < 3:
            continue
        poly_norm = [[float(x) / W, float(y) / H] for x, y in poly_px]
        area_ratio = portion_mod.polygon_area_normalized(poly_norm)
        detections.append(Detection(
            bbox_xyxy=[float(v) for v in box],
            score=float(score),
            mask_area_ratio=float(area_ratio),
            polygon_xy_norm=poly_norm,
        ))

    return detections


def _polygon_to_binary_mask(poly_norm, H: int, W: int) -> np.ndarray:
    """Rasterize a normalized polygon to a HxW boolean mask."""
    cv2 = _cv2()
    mask = np.zeros((H, W), dtype=np.uint8)
    if len(poly_norm) >= 3:
        pts = np.array([[int(x * W), int(y * H)] for x, y in poly_norm], dtype=np.int32)
        cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)


def _run_coco_detector(coco_model, image_rgb: np.ndarray) -> List[Detection]:
    """Run COCO-pretrained YOLO seg model. Return Detection objects with
    labels already filled in (label_source='coco'), keeping only food classes."""
    cv2 = _cv2()
    H, W = image_rgb.shape[:2]
    img_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    pred = coco_model.predict(
        source=img_bgr,
        conf=COCO_CONF_THRESHOLD,
        iou=COCO_IOU_THRESHOLD,
        verbose=False,
    )[0]

    detections: List[Detection] = []
    if pred.boxes is None or len(pred.boxes) == 0:
        return detections

    boxes = pred.boxes.xyxy.cpu().numpy()
    cls_ids = pred.boxes.cls.cpu().numpy().astype(int)
    confs = pred.boxes.conf.cpu().numpy()
    masks_xy = list(pred.masks.xy) if pred.masks is not None else [None] * len(boxes)

    for box, cls_id, conf, mask_xy in zip(boxes, cls_ids, confs, masks_xy):
        if cls_id not in COCO_FOOD_CLASSES:
            continue
        coco_label = COCO_FOOD_CLASSES[cls_id]
        nutrition_key = COCO_TO_NUTRITION_KEY.get(coco_label, coco_label)

        if mask_xy is not None and len(mask_xy) >= 3:
            poly_norm = [[float(x) / W, float(y) / H] for x, y in mask_xy]
        else:
            # Fall back to bbox corners as polygon
            x1, y1, x2, y2 = box
            poly_norm = [
                [float(x1) / W, float(y1) / H],
                [float(x2) / W, float(y1) / H],
                [float(x2) / W, float(y2) / H],
                [float(x1) / W, float(y2) / H],
            ]
        area_ratio = portion_mod.polygon_area_normalized(poly_norm)

        det = Detection(
            bbox_xyxy=[float(v) for v in box],
            score=float(conf),
            mask_area_ratio=float(area_ratio),
            polygon_xy_norm=poly_norm,
        )
        det.label = nutrition_key
        det.label_source = "coco"
        det.ingredient_top = []
        det.dish_top = []
        detections.append(det)
    return detections


def _suppress_produce_overlapping_coco(
    coco_dets: List[Detection],
    produce_dets: List[Detection],
    image_shape,
) -> List[Detection]:
    """Drop produce detections that significantly overlap any COCO detection.
    COCO has higher label trust, so on overlap we let COCO win."""
    if not coco_dets or not produce_dets:
        return produce_dets

    H, W = image_shape[:2]
    coco_masks = [_polygon_to_binary_mask(d.polygon_xy_norm, H, W) for d in coco_dets]
    coco_areas = [int(m.sum()) for m in coco_masks]

    survivors: List[Detection] = []
    for p_det in produce_dets:
        p_mask = _polygon_to_binary_mask(p_det.polygon_xy_norm, H, W)
        p_area = int(p_mask.sum())
        if p_area == 0:
            continue

        suppressed = False
        for c_mask, c_area in zip(coco_masks, coco_areas):
            if c_area == 0:
                continue
            inter = int(np.logical_and(p_mask, c_mask).sum())
            if inter == 0:
                continue
            containment = inter / p_area
            iou = inter / max(1, p_area + c_area - inter)
            if (containment >= COCO_VS_PRODUCE_CONTAINMENT
                    or iou >= COCO_VS_PRODUCE_IOU):
                suppressed = True
                break

        if not suppressed:
            survivors.append(p_det)
    return survivors


def dedupe_overlapping_detections(
    detections: List[Detection],
    image_shape,
    containment_threshold: float = 0.70,
    iou_threshold: float = 0.60,
    same_class_only: bool = True,
    class_key=lambda d: getattr(d, "label", None),
) -> List[Detection]:
    """
    Suppress smaller detections that are mostly inside (or strongly overlap with)
    a larger detection.

    Catches the common YOLO failure mode:
      "whole pizza" detection AND "individual slice" detection are both reported,
      causing nutrition totals to double-count.

    Strategy
    --------
    1. Sort detections by mask area, descending (largest first).
    2. For each detection D (kept), suppress every smaller D' where either
         containment(D', D) = area(D ∩ D') / area(D')  >=  containment_threshold
         iou(D', D)                                    >=  iou_threshold
    3. Returns the surviving detections in their original order.

    Parameters
    ----------
    same_class_only : bool
        Only suppress within the same class_key. Use False to dedupe across
        all classes (useful for the binary "produce" detector where every
        detection has the same label).
    class_key : callable
        Function that returns a class identifier for each detection. Default
        uses the `label` attribute. For pre-classification dedup (no labels
        yet), pass `lambda d: 0` so everything is treated as one class.
    """
    n = len(detections)
    if n <= 1:
        return list(detections)

    H, W = image_shape[:2]
    masks = [_polygon_to_binary_mask(d.polygon_xy_norm, H, W) for d in detections]
    areas = np.array([int(m.sum()) for m in masks], dtype=np.int64)
    classes = [class_key(d) for d in detections]

    # Sort indices by area descending — keep the biggest, suppress nested smaller.
    order = sorted(range(n), key=lambda i: -int(areas[i]))

    suppressed = [False] * n
    for outer_pos, i in enumerate(order):
        if suppressed[i] or areas[i] == 0:
            continue
        mi = masks[i]
        ai = int(areas[i])
        for j in order[outer_pos + 1:]:
            if suppressed[j] or areas[j] == 0:
                continue
            if same_class_only and classes[i] != classes[j]:
                continue
            mj = masks[j]
            aj = int(areas[j])
            inter = int(np.logical_and(mi, mj).sum())
            if inter == 0:
                continue
            containment = inter / aj
            iou = inter / max(1, ai + aj - inter)
            if containment >= containment_threshold or iou >= iou_threshold:
                suppressed[j] = True

    return [detections[i] for i in range(n) if not suppressed[i]]


def _crop_with_mask(image_rgb: np.ndarray, det: Detection) -> np.ndarray:
    """
    Crop the bbox region and apply the polygon mask to suppress background.
    Returns an RGB image suitable for classification.
    """
    cv2 = _cv2()
    H, W = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in det.bbox_xyxy]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(W, x2), min(H, y2)
    if x2 <= x1 or y2 <= y1:
        return image_rgb

    full_mask = np.zeros((H, W), dtype=np.uint8)
    pts_px = np.array(
        [[int(x * W), int(y * H)] for x, y in det.polygon_xy_norm],
        dtype=np.int32,
    )
    cv2.fillPoly(full_mask, [pts_px], 255)

    masked = image_rgb.copy()
    masked[full_mask == 0] = 0
    crop = masked[y1:y2, x1:x2]
    return crop


# ----------------------------------------------------------------------
# Public API
# ----------------------------------------------------------------------

class NutriFilePipeline:
    """End-to-end inference object. Initialize once, reuse for many images."""

    def __init__(
        self,
        detector_weights: Path | str = config.PRODUCE_DETECTOR_WEIGHTS,
        ingredient_weights: Path | str = config.INGREDIENT_CLS_WEIGHTS,
        dish_weights: Path | str = config.DISH_CLS_WEIGHTS,
        coco_weights: Path | str | None = None,
        device: Optional[str] = None,
    ):
        YOLO = _yolo()
        self.detector = YOLO(str(detector_weights))

        # Optional: stock COCO-pretrained yolov8s-seg.pt for prepared dishes
        # (pizza, donut, cake, hot_dog, sandwich) that produce detector misses.
        self.coco_detector = None
        if coco_weights is not None and Path(coco_weights).exists():
            self.coco_detector = YOLO(str(coco_weights))

        self.ingredient_clf = None
        self.dish_clf = None

        if Path(ingredient_weights).exists():
            self.ingredient_clf = cls_mod.load_classifier(
                ingredient_weights, device=device
            )
        if Path(dish_weights).exists():
            self.dish_clf = cls_mod.load_classifier(
                dish_weights, device=device
            )

    # ------------------------------------------------------------------
    def run(
        self,
        image_rgb: np.ndarray,
        user_goal: str = "maintenance",
        target_kcal: float = 2000.0,
    ) -> PipelineResult:
        H, W = image_rgb.shape[:2]

        # Produce detector (binary "is this food?", trained on PackEat)
        produce_dets = _run_yolo_seg(self.detector, image_rgb)

        # COCO ensemble (optional): stock yolov8s-seg knows pizza/donut/cake/etc.
        coco_dets: List[Detection] = []
        if self.coco_detector is not None:
            coco_dets = _run_coco_detector(self.coco_detector, image_rgb)

        # ----------------------------------------------------------
        # Pre-classification dedup of produce detections among themselves.
        # The trained produce detector emits only one class ("produce"),
        # so they're all interchangeable -- same_class_only=False treats
        # them as one bucket.
        # ----------------------------------------------------------
        if config.DEDUPE_ENABLED and len(produce_dets) > 1:
            produce_dets = dedupe_overlapping_detections(
                produce_dets,
                image_shape=image_rgb.shape,
                containment_threshold=config.DEDUPE_CONTAINMENT_THRESHOLD,
                iou_threshold=config.DEDUPE_IOU_THRESHOLD,
                same_class_only=False,
                class_key=lambda d: 0,
            )

        # When COCO detected pizza/cake/etc., drop produce detections that
        # overlap with it (COCO has higher label trust).
        produce_dets = _suppress_produce_overlapping_coco(
            coco_dets, produce_dets, image_rgb.shape
        )

        detections = list(coco_dets) + list(produce_dets)

        # ----------------------------------------------------------
        # Per-detection: classify (skip for COCO) + portion + nutrition
        # ----------------------------------------------------------
        for det in detections:
            if det.label_source == "coco":
                # COCO already gave us a confident label; skip the classifier.
                pass
            else:
                crop = _crop_with_mask(image_rgb, det)

                if self.ingredient_clf is not None:
                    det.ingredient_top = cls_mod.predict_topk(self.ingredient_clf, crop, topk=3)
                if self.dish_clf is not None:
                    det.dish_top = cls_mod.predict_topk(self.dish_clf, crop, topk=3)

                det.label, det.label_source = _route_label(det, det.ingredient_top, det.dish_top)

            entry = nutrition.lookup(det.label)
            est = portion_mod.estimate_grams(
                mask_area_ratio=det.mask_area_ratio,
                density_g_per_cm2=float(entry["density_g_per_cm2"]),
            )
            det.portion = est.to_dict()

            grams = est.grams
            det.nutrition = {
                "kcal":    round(entry["kcal"]    * grams / 100, 1),
                "carbs":   round(entry["carbs"]   * grams / 100, 1),
                "protein": round(entry["protein"] * grams / 100, 1),
                "fat":     round(entry["fat"]     * grams / 100, 1),
                "sugar":   round(entry["sugar"]   * grams / 100, 1),
                "sodium":  round(entry["sodium"]  * grams / 100, 1),
                "fiber":   round(entry["fiber"]   * grams / 100, 1),
                "confidence": entry.get("confidence", "low"),
                "source_per_100g": {
                    k: entry[k]
                    for k in ("kcal", "carbs", "protein", "fat",
                              "sugar", "sodium", "fiber")
                },
            }

        # ----------------------------------------------------------
        # Aggregate
        # ----------------------------------------------------------
        agg = {"kcal": 0.0, "carbs": 0.0, "protein": 0.0, "fat": 0.0,
               "sugar": 0.0, "sodium": 0.0, "fiber": 0.0, "total_grams": 0.0}
        for det in detections:
            if det.nutrition is None:
                continue
            agg["kcal"]    += det.nutrition["kcal"]
            agg["carbs"]   += det.nutrition["carbs"]
            agg["protein"] += det.nutrition["protein"]
            agg["fat"]     += det.nutrition["fat"]
            agg["sugar"]   += det.nutrition["sugar"]
            agg["sodium"]  += det.nutrition["sodium"]
            agg["fiber"]   += det.nutrition["fiber"]
            agg["total_grams"] += det.portion["grams"]
        agg = {k: round(v, 1) for k, v in agg.items()}

        # ----------------------------------------------------------
        # Recommendation + explanation
        # ----------------------------------------------------------
        recs = rec_mod.recommendations_for(agg, goal=user_goal, target_kcal=target_kcal)
        text = exp_mod.summarize(detections, agg, recs, goal=user_goal,
                                 target_kcal=target_kcal)

        return PipelineResult(
            image_shape=(H, W, image_rgb.shape[2] if image_rgb.ndim == 3 else 1),
            detections=detections,
            aggregated_nutrition=agg,
            recommendations=recs,
            explanation=text,
        )

'''.lstrip('\n'),
    'recommend.py': r'''
"""
Mesin rekomendasi berbasis aturan.

Setiap aturan berjalan independen dan menghasilkan saran terstruktur:
- code:        identifier yang machine-readable
- severity:    "info" | "warning" | "alert"
- message:     satu baris yang bisa dibaca manusia
- evidence:    angka-angka yang memicu aturan

Mesinnya sengaja dibuat transparan dan gampang diedit.
Siapapun bisa tambah aturan baru tanpa menyentuh bagian pipeline lain.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List


# target per-makan, diturunkan dari anggaran kalori harian
# rasio makronya mengikuti pedoman umum (US/EU)

def per_meal_targets(daily_kcal: float, meals_per_day: int = 3) -> Dict[str, float]:
    meal_kcal = daily_kcal / meals_per_day
    return {
        # macros in grams
        "kcal":    meal_kcal,
        "carbs":   meal_kcal * 0.50 / 4,        # 50 %E carbs, 4 kcal/g
        "protein": meal_kcal * 0.20 / 4,        # 20 %E protein, 4 kcal/g
        "fat":     meal_kcal * 0.30 / 9,        # 30 %E fat, 9 kcal/g
        "sugar":   25.0,                        # WHO upper bound per meal (~75 g/day)
        "sodium":  766.0,                       # 2300 mg/day / 3 meals
        "fiber":   28.0 / meals_per_day,        # ~28 g daily
    }


# ----------------------------------------------------------------------
# Rule signature
# ----------------------------------------------------------------------

@dataclass
class Suggestion:
    code: str
    severity: str
    message: str
    evidence: dict

    def to_dict(self) -> dict:
        return {
            "code": self.code,
            "severity": self.severity,
            "message": self.message,
            "evidence": self.evidence,
        }


# aturan-aturan individual

def _rule_calorie_target(agg, targets, goal) -> List[Suggestion]:
    out: List[Suggestion] = []
    kcal = agg["kcal"]
    tgt = targets["kcal"]

    over = kcal - tgt
    pct_over = (over / tgt) if tgt > 0 else 0

    if goal == "weight_loss":
        if pct_over > 0.10:
            out.append(Suggestion(
                code="CAL_OVER_WEIGHT_LOSS",
                severity="alert",
                message=(
                    f"This meal has ~{kcal:.0f} kcal, "
                    f"{int(pct_over * 100)}% above your weight-loss target of "
                    f"{tgt:.0f} kcal/meal. Consider reducing portion size."
                ),
                evidence={"kcal": kcal, "target": tgt},
            ))
        elif pct_over > -0.10:
            out.append(Suggestion(
                code="CAL_ON_TARGET",
                severity="info",
                message=f"Calories ({kcal:.0f}) are close to your target ({tgt:.0f}).",
                evidence={"kcal": kcal, "target": tgt},
            ))
    elif goal == "muscle_gain":
        if pct_over < -0.10:
            out.append(Suggestion(
                code="CAL_UNDER_MUSCLE",
                severity="warning",
                message=(
                    f"This meal is only ~{kcal:.0f} kcal vs your target of "
                    f"{tgt:.0f} kcal. Add an extra protein or carb side to support muscle gain."
                ),
                evidence={"kcal": kcal, "target": tgt},
            ))
    else:  # maintenance
        if abs(pct_over) > 0.20:
            direction = "above" if pct_over > 0 else "below"
            out.append(Suggestion(
                code="CAL_OFF_MAINTENANCE",
                severity="warning",
                message=(
                    f"This meal is ~{abs(int(pct_over * 100))}% {direction} a typical "
                    f"maintenance meal ({tgt:.0f} kcal)."
                ),
                evidence={"kcal": kcal, "target": tgt},
            ))

    return out


def _rule_protein(agg, targets, goal) -> List[Suggestion]:
    out: List[Suggestion] = []
    p = agg["protein"]
    tgt = targets["protein"]
    if p < tgt * 0.7:
        msg_extra = " Especially important for muscle gain." if goal == "muscle_gain" else ""
        out.append(Suggestion(
            code="LOW_PROTEIN",
            severity="warning",
            message=(
                f"Only {p:.1f} g protein in this meal vs target {tgt:.1f} g. "
                f"Consider adding chicken, fish, eggs, tofu, or beans.{msg_extra}"
            ),
            evidence={"protein": p, "target": tgt},
        ))
    return out


def _rule_sugar(agg, targets, goal) -> List[Suggestion]:
    out: List[Suggestion] = []
    s = agg["sugar"]
    if s > targets["sugar"] * 1.2:
        out.append(Suggestion(
            code="HIGH_SUGAR",
            severity="alert",
            message=(
                f"Sugar is {s:.1f} g — well above the {targets['sugar']:.0f} g per-meal "
                f"guideline. Cut dessert or sweetened items."
            ),
            evidence={"sugar": s, "target": targets["sugar"]},
        ))
    return out


def _rule_sodium(agg, targets, goal) -> List[Suggestion]:
    out: List[Suggestion] = []
    na = agg["sodium"]
    if na > targets["sodium"] * 1.2:
        out.append(Suggestion(
            code="HIGH_SODIUM",
            severity="alert",
            message=(
                f"Sodium is {na:.0f} mg — high for a single meal "
                f"(target <= {targets['sodium']:.0f} mg). Watch out for cured meats, "
                f"sauces, and salty snacks."
            ),
            evidence={"sodium": na, "target": targets["sodium"]},
        ))
    return out


def _rule_fat(agg, targets, goal) -> List[Suggestion]:
    out: List[Suggestion] = []
    f = agg["fat"]
    if f > targets["fat"] * 1.5:
        out.append(Suggestion(
            code="HIGH_FAT",
            severity="warning",
            message=(
                f"Fat is {f:.1f} g — more than 1.5× the per-meal target "
                f"({targets['fat']:.1f} g)."
            ),
            evidence={"fat": f, "target": targets["fat"]},
        ))
    return out


def _rule_fiber(agg, targets, goal) -> List[Suggestion]:
    out: List[Suggestion] = []
    fi = agg["fiber"]
    if fi < targets["fiber"] * 0.5 and agg["kcal"] > 200:
        out.append(Suggestion(
            code="LOW_FIBER",
            severity="info",
            message=(
                f"Only {fi:.1f} g fiber. Add vegetables, fruit, or whole-grain bread to "
                f"reach the per-meal target (~{targets['fiber']:.1f} g)."
            ),
            evidence={"fiber": fi, "target": targets["fiber"]},
        ))
    return out


def _rule_balance(agg, targets, goal) -> List[Suggestion]:
    out: List[Suggestion] = []
    kcal = agg["kcal"]
    if kcal <= 0:
        return out
    carbs_pct = (agg["carbs"] * 4) / kcal
    prot_pct  = (agg["protein"] * 4) / kcal
    fat_pct   = (agg["fat"] * 9) / kcal

    if carbs_pct > 0.65:
        out.append(Suggestion(
            code="CARB_DOMINANT",
            severity="info",
            message=(
                f"This meal is carb-heavy ({carbs_pct:.0%} of calories from carbs). "
                f"Balance with protein and vegetables."
            ),
            evidence={"carbs_pct": round(carbs_pct, 3)},
        ))
    if fat_pct > 0.45:
        out.append(Suggestion(
            code="FAT_DOMINANT",
            severity="info",
            message=(
                f"This meal is fat-heavy ({fat_pct:.0%} of calories from fat)."
            ),
            evidence={"fat_pct": round(fat_pct, 3)},
        ))
    if prot_pct < 0.10 and kcal > 200:
        out.append(Suggestion(
            code="LOW_PROTEIN_PCT",
            severity="warning",
            message=(
                f"Protein supplies only {prot_pct:.0%} of energy. "
                f"Add a protein source."
            ),
            evidence={"protein_pct": round(prot_pct, 3)},
        ))

    return out


# jalankan semua aturan

RULES = [
    _rule_calorie_target,
    _rule_protein,
    _rule_sugar,
    _rule_sodium,
    _rule_fat,
    _rule_fiber,
    _rule_balance,
]


def recommendations_for(
    aggregated_nutrition: dict,
    goal: str = "maintenance",
    target_kcal: float = 2000.0,
    meals_per_day: int = 3,
) -> List[dict]:
    """Jalankan semua aturan dan return list saran dalam bentuk dict."""
    targets = per_meal_targets(target_kcal, meals_per_day=meals_per_day)

    out: List[Suggestion] = []
    for rule in RULES:
        out.extend(rule(aggregated_nutrition, targets, goal))

    return [s.to_dict() for s in out]

'''.lstrip('\n'),
    'explain.py': r'''
"""
Bikin ringkasan makanan yang bisa dibaca manusia.

Input: daftar deteksi + nutrisi agregat + saran.
Output: teks ringkasan untuk UI aplikasi.
"""

from __future__ import annotations

from typing import List


def _format_label(label: str) -> str:
    if not label:
        return "unknown"
    return label.replace("_", " ")


def summarize(
    detections: List,
    aggregated: dict,
    recommendations: List[dict],
    goal: str = "maintenance",
    target_kcal: float = 2000.0,
) -> str:
    lines: List[str] = []

    # ----- Composition -----
    if not detections:
        lines.append("No food items were detected in the image.")
    else:
        lines.append(f"Detected {len(detections)} food region(s):")
        for i, det in enumerate(detections, start=1):
            label_pretty = _format_label(det.label or "unknown")
            grams = det.portion["grams"] if det.portion else 0.0
            kcal  = det.nutrition["kcal"] if det.nutrition else 0.0
            src   = det.label_source or "unknown"
            lines.append(
                f"  {i}. {label_pretty}  ~{grams:.0f} g  ({kcal:.0f} kcal, via {src} head)"
            )

    # ----- Totals -----
    lines.append("")
    lines.append("Estimated total nutrition for this meal:")
    lines.append(
        f"  Calories : {aggregated['kcal']:.0f} kcal\n"
        f"  Carbs    : {aggregated['carbs']:.1f} g\n"
        f"  Protein  : {aggregated['protein']:.1f} g\n"
        f"  Fat      : {aggregated['fat']:.1f} g\n"
        f"  Sugar    : {aggregated['sugar']:.1f} g\n"
        f"  Sodium   : {aggregated['sodium']:.0f} mg\n"
        f"  Fiber    : {aggregated['fiber']:.1f} g\n"
        f"  (≈ {aggregated['total_grams']:.0f} g total estimated food weight)"
    )

    lines.append("")
    lines.append(f"Goal: {goal} | Daily target: {target_kcal:.0f} kcal")

    # ----- Recommendations -----
    if not recommendations:
        lines.append("Recommendations: meal looks well-balanced for your goal.")
    else:
        lines.append("Recommendations:")
        for rec in recommendations:
            sev = rec["severity"].upper()
            lines.append(f"  [{sev}] {rec['message']}")

    return "\n".join(lines)

'''.lstrip('\n'),
}
for name, src in FILES.items():
    (PKG / name).write_text(src, encoding='utf-8')
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import nutrifile
print(nutrifile.config.summary())

## Step 1 - Load pretrained COCO YOLOv8-seg

In [ ]:
%pip install -q ultralytics

import torch
from ultralytics import YOLO

# This downloads ~22 MB from Ultralytics on first run.
coco_model = YOLO("yolov8s-seg.pt")

print("Device available:", "cuda" if torch.cuda.is_available() else "cpu")
print("Model classes :", len(coco_model.names))

# COCO id -> canonical nutrition DB key used by NutriFile
COCO_FOOD_MAP = {
    46: "banana",
    47: "apple",
    48: "club_sandwich",   # closest dish in our DB
    49: "lemon",           # COCO "orange" - use lemon as proxy
    50: "broccoli",
    51: "carrot",
    52: "hot_dog",
    53: "pizza",
    54: "donuts",
    55: "chocolate_cake",  # closest dish in our DB
}
print("Food classes recognized in smoke mode:", list(COCO_FOOD_MAP.values()))


## Step 2 - Grab a test image
Tries (in order): (a) a path you set in `TEST_IMAGE_PATH`, (b) PackEat test split if you've already prepared it, (c) a fallback public food image URL.

In [ ]:
import urllib.request
from pathlib import Path
import cv2
import numpy as np
from nutrifile import config

TEST_IMAGE_PATH = None      # <-- you may set this to your own image path

def load_test_image():
    # Option A: user-provided path
    if TEST_IMAGE_PATH and Path(TEST_IMAGE_PATH).exists():
        img = cv2.imread(str(TEST_IMAGE_PATH))
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), str(TEST_IMAGE_PATH)

    # Option B: PackEat test image if you've run notebook 01
    pack_dir = config.PRODUCE_SEG_DIR / "images" / "test"
    if pack_dir.exists():
        candidates = sorted(pack_dir.glob("*.jpg"))
        if candidates:
            img = cv2.imread(str(candidates[0]))
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), str(candidates[0])

    # Option C: download Ultralytics' bus.jpg - wait, we want food. Use a public pizza pic.
    url = "https://ultralytics.com/images/bus.jpg"           # works but not food
    food_urls = [
        "https://images.unsplash.com/photo-1565299624946-b28f40a0ae38?w=800",  # pizza
        "https://images.unsplash.com/photo-1546069901-ba9599a7e63c?w=800",     # salad
    ]
    for url in food_urls:
        try:
            tmp = config.WORK_ROOT / "smoke_test_image.jpg"
            urllib.request.urlretrieve(url, tmp)
            img = cv2.imread(str(tmp))
            if img is not None:
                return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), str(tmp)
        except Exception as e:
            print(f"  failed to fetch {url}: {e}")
    raise RuntimeError("Could not obtain a test image. Set TEST_IMAGE_PATH manually.")

image_rgb, source = load_test_image()
print(f"Loaded image from: {source}")
print(f"Shape: {image_rgb.shape}")

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 8))
plt.imshow(image_rgb)
plt.title("Test image")
plt.axis("off")
plt.show()


## Step 3 - Run detection and visualize raw output
Sanity check: do we see boxes/masks on the food?

In [ ]:
import cv2
import matplotlib.pyplot as plt

img_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
res = coco_model.predict(source=img_bgr, conf=0.25, iou=0.55, verbose=False)[0]

n_obj = 0 if res.boxes is None else len(res.boxes)
print(f"Total raw detections: {n_obj}")

if res.boxes is not None:
    for i in range(len(res.boxes)):
        cls_id = int(res.boxes.cls[i].item())
        conf   = float(res.boxes.conf[i].item())
        name   = coco_model.names[cls_id]
        is_food = cls_id in COCO_FOOD_MAP
        print(f"  det {i}: {name:18s}  conf={conf:.2f}  food={is_food}")

annotated = res.plot()
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("COCO YOLOv8-seg raw output")
plt.axis("off")
plt.show()


## Step 4 - Run the full pipeline end-to-end
Portion estimation + nutrition lookup + recommendation + explanation.

In [ ]:
"""
Manually do what NutriFilePipeline does, but with the pretrained COCO model
acting as the produce detector and the COCO class name acting as the label.

Includes the SAME nested-detection dedup the trained pipeline uses, so e.g.
a pizza detected as both "whole pizza" and "individual slices" collapses to
a single instance and totals don't double-count.
"""
import cv2
import numpy as np
from dataclasses import dataclass, field
from typing import List
from nutrifile import portion as portion_mod
from nutrifile import nutrition, recommend, explain
from nutrifile import config as nutri_cfg
from nutrifile.pipeline import dedupe_overlapping_detections

H, W = image_rgb.shape[:2]


# Build Detection-shaped objects so we can reuse dedupe_overlapping_detections.
@dataclass
class _ShimDet:
    polygon_xy_norm: list
    label: str
    score: float
    bbox: list
    area_ratio: float


raw_dets = []
if res.boxes is not None:
    for i in range(len(res.boxes)):
        cls_id = int(res.boxes.cls[i].item())
        if cls_id not in COCO_FOOD_MAP:
            continue
        conf  = float(res.boxes.conf[i].item())
        box   = res.boxes.xyxy[i].cpu().numpy().tolist()
        poly  = res.masks.xy[i] if res.masks is not None else None
        if poly is None or len(poly) < 3:
            continue
        poly_norm = [[float(x) / W, float(y) / H] for x, y in poly]
        area_ratio = portion_mod.polygon_area_normalized(poly_norm)
        raw_dets.append(_ShimDet(
            polygon_xy_norm=poly_norm,
            label=COCO_FOOD_MAP[cls_id],
            score=conf, bbox=box, area_ratio=area_ratio,
        ))

# Apply the SAME dedup the trained pipeline uses.
print(f"Raw food detections (before dedup): {len(raw_dets)}")
for d in raw_dets:
    print(f"  RAW   {d.label:18s}  area={d.area_ratio:.2%}  conf={d.score:.2f}")

if nutri_cfg.DEDUPE_ENABLED and len(raw_dets) > 1:
    deduped = dedupe_overlapping_detections(
        raw_dets,
        image_shape=image_rgb.shape,
        containment_threshold=nutri_cfg.DEDUPE_CONTAINMENT_THRESHOLD,
        iou_threshold=nutri_cfg.DEDUPE_IOU_THRESHOLD,
        same_class_only=nutri_cfg.DEDUPE_SAME_CLASS_ONLY,
    )
else:
    deduped = list(raw_dets)

print(f"\nFood detections after dedup: {len(deduped)}  "
      f"(removed {len(raw_dets) - len(deduped)})")

# Now compute portion + nutrition on the deduped set.
food_dets = []
for d in deduped:
    entry = nutrition.lookup(d.label)
    est   = portion_mod.estimate_grams(
        mask_area_ratio=d.area_ratio,
        density_g_per_cm2=float(entry["density_g_per_cm2"]),
    )
    grams = est.grams
    per_g = grams / 100
    det_nutrition = {
        "kcal":    round(entry["kcal"]    * per_g, 1),
        "carbs":   round(entry["carbs"]   * per_g, 1),
        "protein": round(entry["protein"] * per_g, 1),
        "fat":     round(entry["fat"]     * per_g, 1),
        "sugar":   round(entry["sugar"]   * per_g, 1),
        "sodium":  round(entry["sodium"]  * per_g, 1),
        "fiber":   round(entry["fiber"]   * per_g, 1),
    }
    food_dets.append({
        "label": d.label, "score": d.score, "bbox": d.bbox,
        "poly_norm": d.polygon_xy_norm, "area_ratio": d.area_ratio,
        "grams": grams, "nutrition": det_nutrition,
    })

print()
for d in food_dets:
    print(f"  KEPT  {d['label']:18s}  area={d['area_ratio']:.2%}  "
          f"~{d['grams']:.0f} g  {d['nutrition']['kcal']:.0f} kcal")

agg = {"kcal": 0.0, "carbs": 0.0, "protein": 0.0, "fat": 0.0,
       "sugar": 0.0, "sodium": 0.0, "fiber": 0.0, "total_grams": 0.0}
for d in food_dets:
    for k in ("kcal", "carbs", "protein", "fat", "sugar", "sodium", "fiber"):
        agg[k] += d["nutrition"][k]
    agg["total_grams"] += d["grams"]
agg = {k: round(v, 1) for k, v in agg.items()}

print("\nAggregated:", agg)

recs = recommend.recommendations_for(agg, goal="maintenance", target_kcal=2000)
print(f"\nRecommendations: {len(recs)} rules fired")
for r in recs:
    print(f"  [{r['severity']:<7}] {r['code']:<22} {r['message']}")

# Build a FakeDet-shape for the explainer
from dataclasses import dataclass
@dataclass
class FakeDet:
    label: str; label_source: str; portion: dict; nutrition: dict

fake_dets = [FakeDet(
    label=d["label"], label_source="coco_smoke",
    portion={"grams": d["grams"]}, nutrition=d["nutrition"],
) for d in food_dets]

print("\n" + "=" * 80)
print(explain.summarize(fake_dets, agg, recs, goal="maintenance", target_kcal=2000))


## Step 5 - Sanity assertions
These are the same checks the unit tests run, but on real model output.

In [ ]:
"""
Hard assertions that fail loudly if any pipeline stage produced nonsense.
If everything passes, your wiring is correct.
"""

errors = []

# Detection happened
if len(food_dets) == 0:
    errors.append("No food detections - either the test image has no COCO food classes "
                  "or YOLO confidence is too low. Try lowering conf=0.25 to 0.15 above.")

# Sane grams
for d in food_dets:
    if d["grams"] <= 0 or d["grams"] > 2000:
        errors.append(f"Implausible grams for {d['label']}: {d['grams']:.1f}")

# Sane calories
if food_dets and (agg["kcal"] <= 0 or agg["kcal"] > 5000):
    errors.append(f"Implausible total kcal: {agg['kcal']}")

# Nutrition fields present
for d in food_dets:
    missing = {"kcal","carbs","protein","fat","sugar","sodium","fiber"} - set(d["nutrition"])
    if missing:
        errors.append(f"{d['label']} missing nutrition fields: {missing}")

# Recommendation engine returned a list of dicts with expected schema
for r in recs:
    if not {"code","severity","message","evidence"} <= set(r):
        errors.append(f"Bad recommendation schema: {r}")

if errors:
    print("SMOKE TEST FAILED:")
    for e in errors:
        print("  -", e)
else:
    print("SMOKE TEST PASSED.")
    print(f"  detections     : {len(food_dets)}")
    print(f"  total grams    : {agg['total_grams']:.0f}")
    print(f"  total kcal     : {agg['kcal']:.0f}")
    print(f"  recommendations: {len(recs)}")
    print("\nThe full pipeline is wired correctly. You can now proceed with the")
    print("training notebooks (01-04) and re-run the inference demo (05).")
